# 因子選股策略回測：完整流程（取得資料 → Phase 1~4 SOP）

從資料庫取資料開始，依序執行 Phase 1（單因子健檢）→ Phase 2（F1×F2配對）→
Phase 3（加動態條件C）→ Phase 4（加估值濾網V）。

程式邏輯跟 `code/phase1_*.py`~`phase4_*.py` 完全一致（本notebook呼叫那些程式，
不是另外重寫一份），跑出來的結果跟直接用命令列執行相同。

由上而下依序執行每個 code cell 即可。

⚠️ Phase 1 較快（幾分鐘），Phase 2~4 隨組合數增加，可能要數十分鐘到數小時。
每個真跑的cell前面都會先跑一次 `--dry-run` 看預期跑幾個策略，再決定要不要往下跑真的。


## 0. 環境設定

`code/` 底下的程式需要 import 到上層目錄的 `database.py`、`get_data.py` 等模組。


In [ ]:
import sys
sys.path.insert(0, '../')

import warnings
warnings.filterwarnings('ignore')

print('目前工作目錄應為 code/ ：', __import__('os').getcwd())


## 1. 取得資料

回測引擎透過 `fcv_core.MarketData` 一次載入某個市場的價量+財報資料，
Phase 1~4 重複使用這份已載入的資料。

`MarketData` 做的事：
1. 呼叫 `get_data.Data(market=...)` 從資料庫抓股價、財報、因子原始資料。
2. 回測宇宙＝「有因子資料」∩「有股價資料」的股票交集。
3. 依 `start`/`end` 參數把資料裁到 in-sample 範圍（2000-01-01~2025-12-31，
   2026年之後保留當作樣本外，不用來訓練/篩選，避免用到未來資訊）。

台股示範（美股同樣方式，把 `market='TW'` 換成 `market='US'`）：


In [ ]:
from fcv_core import MarketData

md_demo = MarketData(market='TW', end='2025-12-31')

close = md_demo.data.get('price:close')
print('股價資料形狀（交易日 x 股票數）：', close.shape)
close.tail()


In [ ]:
roe = md_demo.get_field('report:ROE')
print('ROE 資料形狀：', roe.shape if roe is not None else None)

md_demo.release()   # 釋放快取


⚠️ 下面 Phase 1~4 的每個腳本都是各自獨立可執行的程式，執行時會**各自重新呼叫一次
`MarketData` 重新載入資料**，不會共用上面這個demo載入的 `md_demo`。


## 2. Phase 1：單因子線性檢定

每個候選因子單獨切成 9 個分位桶（q_band），不搭配其他因子、不加動態條件，
檢查報酬是否隨分位單調變化（Spearman ρ）。未通過線性檢定的因子不會用於後續
Phase 2~4 的 primary 因子。

台股因子池 20 個（PE 保留給 Phase 4 估值濾網使用，不在此因子池內）。
先 `--dry-run` 看預期跑幾個策略：


In [ ]:
%run phase1_linearity.py --market TW --dry-run


確認數字合理後，拿掉 `--dry-run` 真的執行（9桶×20因子×2個V模式，約數百個策略，
通常數分鐘內可完成）：


In [ ]:
%run phase1_linearity.py --market TW


用 `phase1_analyze.py` 分析健檢結果——算每個因子的 Spearman ρ、判定有沒有線性、
畫出9桶報酬曲線圖，輸出到 `../_analysis_outputs_phase1/`：


In [ ]:
%run phase1_analyze.py --market TW


看結果——每個因子的線性判定表，跟9桶報酬曲線圖：

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('../_analysis_outputs_phase1/TW_phase1_linearity.csv')
display(df)
display(Image('../_analysis_outputs_phase1/TW_phase1_curves.png'))


## 3. Phase 2：F1×F2 不對稱配對

Phase 1 通過線性檢定的因子分成 primary（嚴格門檻）／secondary（寬鬆）。
`--variant openSec`：primary 用嚴格標準篩，secondary 全部開放。


In [ ]:
%run phase2_pairing.py --market TW --variant openSec --dry-run


拿掉 `--dry-run` 真的執行（F1×F2 兩兩配對，策略數比 Phase 1 多，通常數十分鐘等級）：


In [ ]:
%run phase2_pairing.py --market TW --variant openSec


In [ ]:
%run phase2_analyze.py --market TW --variant openSec


看結果——配對後的體質檢查表（前10筆），跟配對增益分布圖：

In [ ]:
df = pd.read_csv('../_analysis_outputs_phase2/TW_L2_openSec_體質檢查表.csv')
display(df.head(10))
display(Image('../_analysis_outputs_phase2/TW_L2_openSec_pairing_gain.png'))


## 4. Phase 3：加入動態條件 C

在 Phase 2 通過配對的 F1×F2 策略上，疊加 20 種動態條件 C（衍生自 ROE/EPS/FCF_P，
例如「較上季升」「近N季最高」等時序條件），測試加入時間點篩選後績效是否提升。


In [ ]:
%run phase3_conditions.py --market TW --variant openSec --dry-run


拿掉 `--dry-run` 真的執行（策略數再乘上20種C條件，是四個階段中最耗時的一段，
可能要數小時）：


In [ ]:
%run phase3_conditions.py --market TW --variant openSec


In [ ]:
%run phase3_analyze.py --market TW --variant openSec


看結果——加C後的候選策略表（前10筆，依CAGR排序），跟C條件增益圖：

In [ ]:
df = pd.read_csv('../_analysis_outputs_phase3/TW_L3_openSec_candidate_strategies.csv')
display(df.sort_values('CAGR', ascending=False).head(10))
display(Image('../_analysis_outputs_phase3/TW_L3_openSec_C_gain.png'))


**圖4-16**：控制 F 之後，各 C 條件的 CAGR 折線圖（左：12個單因子組合，右：全部203個F組合）。
直看＝在這個F區間下哪個C最好；橫看＝這個C是不是穩定有效：


In [ ]:
%run phase3_fig416.py --market TW


In [ ]:
display(Image('../_analysis_outputs_phase3/TW_L3_fig4-16_controlled_C_lines.png'))


## 5. Phase 4：加入估值濾網 V

疊加 PE 相對估值濾網（v0=不濾、v1=濾），測試估值維度是否能再提升績效。
FCV 框架（F體質因子 × C動態條件 × V估值濾網）的最後一個維度：


In [ ]:
%run phase4_valuation.py --market TW --variant openSec --dry-run


拿掉 `--dry-run` 真的執行：


In [ ]:
%run phase4_valuation.py --market TW --variant openSec


In [ ]:
%run phase4_analyze.py --market TW --variant openSec


看結果——最終候選策略表（前10筆，依CAGR排序），跟V濾網效果圖：

In [ ]:
df = pd.read_csv('../_analysis_outputs_phase4/TW_L4_openSec_final_candidates.csv')
display(df.sort_values('CAGR', ascending=False).head(10))
display(Image('../_analysis_outputs_phase4/TW_L4_openSec_V_effect.png'))


## 6. 完整圖鑑（18張圖）

`build_atlas.py` 把 Phase 3（v0）與 Phase 4（v1）的結果合併，產出一組共18張圖的完整圖鑑，
涵蓋策略貢獻分解、F1×F2熱力圖、C條件分布、V0/V1比較等面向。需要 Phase 3、Phase 4 都已執行完：


In [ ]:
%run build_atlas.py --market TW --variant openSec


In [ ]:
from pathlib import Path

fig_dir = Path('../_analysis_outputs_atlas/TW_openSec/figures')
for p in sorted(fig_dir.glob('*.png')):
    print(p.name)
    display(Image(str(p)))


## 7. 跑完之後

- 每個 Phase 的分析結果都在 `../_analysis_outputs_phase{1,2,3,4}/`，可以跟隨附的
  「參考結果」資料夾對照數字是否一致（同一份程式+同一份資料庫，Phase1-4是確定性
  流程，理論上應完全一致）。
- **美股**：把上面每個 cell 裡的 `--market TW` 換成 `--market US` 即可，流程相同。
